In [27]:
import pandas as pd
import random

def read_fasta(filepath):
    records = []
    header = None
    seq = []

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()

            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq)))
                header = line[1:]
                seq = []
            else:
                seq.append(line)

        if header is not None:
            records.append((header, "".join(seq)))

    # extract UniProt ID
    parsed = []
    for h, s in records:
        parts = h.split("|")
        uniprot_id = parts[1] if len(parts) > 1 else h.split()[0]
        parsed.append((uniprot_id, s))

    return parsed


# Load toxins
toxins = read_fasta("./data/Toxins.fasta")
n_toxins = len(toxins)

# Load non-toxins
non_toxins = read_fasta("./data/Non-toxins.fasta")

if len(non_toxins) < n_toxins:
    raise ValueError("Not enough non-toxin sequences to match toxin count.")

# Sample equal number
non_toxins_subset = random.sample(non_toxins, n_toxins)

# Combine
all_data = toxins + non_toxins_subset

data = pd.DataFrame({
    "uniprot_id": [x[0] for x in all_data],
    "sequence": [x[1] for x in all_data],
    "toxicity": [1]*n_toxins + [0]*n_toxins
})
data.head()

,uniprot_id,sequence,toxicity
0,A0S864,MKTLLLAVAVVAFVCLGSADQLGLGRQQIDWGQGQAVGPPYTLCFE...,1
1,A0S865,MKTLLLAVAVVAFVCLGSADQLGLGRQQIDWGKGQAKGPPYTLCFE...,1
2,B3EWF8,MKTLLLTLVVVTIVCLDLGNSLKCYVSREGKTQTCPEGEKLCEKYA...,1
3,C0HJD3,QAIGPPFGLCFQCNQKTSSDCFNAKRCPPFHRTCYTLYKPDGGEEW...,1
4,P01391,IRCFITPDITSKDCPNGHVCYTKTWCDAFCSIRGKRVDLGCAATCP...,1


In [30]:
import os, json
import requests
import pandas as pd
from Bio.PDB import PDBParser
from io import StringIO
from DPFunc_fork.DPFunctional import get_GO_terms

parser = PDBParser(QUIET=True)
output_file = "go_terms_results.json"

# resume: read already-done IDs
done = set()
if os.path.exists(output_file):
    with open(output_file, "r") as f:
        for line in f:
            done.add(json.loads(line)["uniprot_id"])

with open(output_file, "a") as f:
    for uniprot_id in data["uniprot_id"]:
        if uniprot_id in done:
            continue

        af = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}").json()
        pdb_string = requests.get(af[0]["pdbUrl"]).text

        structure = parser.get_structure("protein", StringIO(pdb_string))
        model = structure[0]

        GO_terms = get_GO_terms(model, uniprot_id, debug=True).predictions

        f.write(json.dumps({"uniprot_id": uniprot_id, "GO_terms": GO_terms}) + "\n")
        f.flush()

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/mateo/miniconda3/envs/SafeFold/lib/python3.8/site-packages/IPython/core/interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_13536/4149942060.py", line 6, in <module>
    from DPFunc_fork.DPFunctional import get_GO_terms
  File "/mnt/c/Users/mateo/OneDrive - Imperial College London/Documents/PC Documents/Projects/SafySafy/SafeFold/DPFunc_fork/DPFunctional.py", line 13, in <module>
    from .DPFunc_pred import dpfunc_predict_in_memory
  File "/mnt/c/Users/mateo/OneDrive - Imperial College London/Documents/PC Documents/Projects/SafySafy/SafeFold/DPFunc_fork/DPFunc_pred.py", line 9, in <module>
    from .DPFunc.model_utils import test_performance_gnn_inter, merge_result
  File "/mnt/c/Users/mateo/OneDrive - Imperial College London/Documents/PC Documents/Projects/SafySafy/SafeFold/DPFunc_fork/DPFunc/model_utils.py", line 20, in <module>
    from DPFunc.objective import Aver